<a href="https://colab.research.google.com/github/olucasaguiar/estudos-sobre-machine-learning/blob/main/temas/natural-language-processing/NLP_lucas_nascimento_aguiar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas pyreadstat
import pandas as pd

dataset = r"docs/04832 PERCEPÇÃO DOS BRASILEIROS ACERCA DA DEMOCRACIA/04832.SAV"

df = pd.read_spss(dataset)
(df.info())

Note: you may need to restart the kernel to use updated packages.
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   SEXO             2000 non-null   category
 1   IDADE            2000 non-null   float64 
 2   FX_ID            2000 non-null   category
 3   ESCOLARIDADE     2000 non-null   category
 4   P1A              2000 non-null   category
 5   P1B              2000 non-null   category
 6   P1C              2000 non-null   category
 7   P2_1             2000 non-null   category
 8   P2_2             1883 non-null   category
 9   P2_3             1791 non-null   category
 10  P3_1             2000 non-null   category
 11  P3_2             818 non-null    category
 12  P3_3             304 non-null    category
 13  P3_4             88 non-null     category
 14  P3_5             55 non-null     category
 15  P3_6             41 non-null     c

In [ ]:
scale_features = ['IDADE']

one_k_features = ['SEXO', 'RACA', 'RELIGIAO']
one_k_categories = [df[i].cat.categories for i in one_k_features]

ordinal_features = ['ESCOLARIDADE', 'REND1', 'REND2', 'REGIAO']
ordinal_categories = [df[i].cat.categories for i in ordinal_features]

selected_features = one_k_features + scale_features + ordinal_features

targets = ['P2_1']

In [ ]:
import scipy.stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import  cross_validate, StratifiedKFold, train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MaxAbsScaler
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np


rng = np.random.RandomState(0)
X_train, X_test, y_train, y_test = train_test_split(df[selected_features], df[targets], test_size=0.2, random_state=rng)

## Preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('scaling', MaxAbsScaler(), scale_features),
        ('one_k', OneHotEncoder(categories=one_k_categories, drop='first', sparse_output=False), one_k_features),
        ('ordinal', OrdinalEncoder(categories=ordinal_categories), ordinal_features),
    ],
    remainder='drop',
)

## Feature selection
threshold = VarianceThreshold(threshold=(.8 * (1 - .8)))
kbest = SelectKBest(f_classif, k=int(2e4))
feature_selection = Pipeline(
    steps=[
        ('threshold', threshold),
        ('kbest', kbest),
    ],
)

## Classifier
rfc = RandomForestClassifier(n_jobs=-1, random_state=rng,)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=rng)
parameters = {
    'n_estimators': scipy.stats.randint(100, 250),
    'ccp_alpha': scipy.stats.loguniform(1e-2, 1e0),
    'max_depth': scipy.stats.randint(3, 15),
    'max_features': ('sqrt', 'log2'),
}
search = RandomizedSearchCV(rfc, param_distributions=parameters, cv=cv, n_iter=5, random_state=rng)
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('feature_selection', feature_selection),
        ('search', search),
    ],
    verbose=True,
)
pipeline.fit(X_train, y_train, )
print('score: ', search.best_score_, 'best params: ', search.best_params_)
# scores = cross_validate(pipeline, df[selected_features], df[targets], cv=cv)
# print(scores['test_score'])
# pipeline.fit(X_train, y_train)
# y_pred = pipeline.predict(X_test)

[Pipeline] ...... (step 1 of 3) Processing preprocessor, total=   0.0s
[Pipeline] . (step 2 of 3) Processing feature_selection, total=   0.0s


/home/lucasaguiar/projects/projeto-simulacao-opiniao-publica/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=20000 is greater than n_features=8. All the features will be returned.
  warnings.warn(
/home/lucasaguiar/projects/projeto-simulacao-opiniao-publica/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/lucasaguiar/projects/projeto-simulacao-opiniao-publica/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/lucasaguiar/projects/projeto-simulacao-opiniao-publica/.venv/lib/python3.12/site-packages/sklearn/b

[Pipeline] ............ (step 3 of 3) Processing search, total=  15.5s
score:  0.20125 best params:  {'ccp_alpha': np.float64(0.7727961198545303), 'max_depth': 7, 'max_features': 'log2', 'n_estimators': 183}


In [ ]:
%pip install matplotlib
from sklearn.metrics import ConfusionMatrixDisplay
print (pipeline.feature_names_in_)
# y_pred = pipeline.predict(X_test)
# disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
# print(classification_report(y_test, y_pred))

Note: you may need to restart the kernel to use updated packages.
['SEXO' 'RACA' 'RELIGIAO' 'IDADE' 'ESCOLARIDADE' 'REND1' 'REND2' 'REGIAO']
